# Data Loading & Reformatting

In this notebook, we'll download the Metacritic data set from Kaggle and check whether everything is represented correctly.  
We will also reformat the data frame. The original data contains ~300 variables that are a flattened representation of the system-dependent user review counts from the corresponding game. This should be reformatted to be made usable.

## Setup

In [ ]:
# +++ Import necessary modules +++

import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from core.data import load_from_kaggle, overview


In [2]:
# +++ Set globals +++
PROJECT_ROOT = Path.cwd().parent
RAW_DAT_DIR = PROJECT_ROOT / 'data' / 'raw'

## Load and save data set

In [3]:
# +++ Get data set from kaggle website and save to raw folder +++

full_link = r"https://www.kaggle.com/datasets/zaireali/metacritic-games-scrape"
dataset_link = full_link.split("/datasets/")[-1]

destination = '../data/raw'
dataset_name = dataset_link.split('/')[1]

print(f'📦 Loading data set: {dataset_name}')
files = load_from_kaggle(dataset_link = dataset_link,
                         destination = destination)

print(f'✅ {len(files)} File(s) found:')
for i, file in enumerate(files, 1):
    print(f"   {i}. {file}")

📦 Loading data set: metacritic-games-scrape


100%|██████████| 5.04M/5.04M [00:00<00:00, 6.15MB/s]

Extracting files...
Loading dataset from C:\Users\janos\.cache\kagglehub\datasets\zaireali\metacritic-games-scrape\versions\2 to ../data/raw\metacritic-games-scrape
Moving file: C:\Users\janos\.cache\kagglehub\datasets\zaireali\metacritic-games-scrape\versions\2\dataset_metacritic_scraper_2025-02-15.csv to c:\Users\janos\Projects\StackFuel_PP\notebooks\../data/raw\metacritic-games-scrape
Files moved to '../data/raw\metacritic-games-scrape' directory.
✅ 1 File(s) found:
   1. dataset_metacritic_scraper_2025-02-15.csv


## Read data and do first inspection

In [4]:
# +++ Load data into workspace +++

df = pd.read_csv(RAW_DAT_DIR / files[0])

<positron-console-cell-4>:3: DtypeWarning: Columns (2,155,162,163,170,171,178,179,186,195,196,197,198) have mixed types. Specify dtype option on import or set low_memory=False.


In [5]:
# some meta information about the data set

print(f"\n🔢 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🔄 Duplicates: {df.duplicated().sum():,} ({df.duplicated().sum()/len(df)*100:.2f}%)")
print(f"💾 Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


🔢 Shape: 13,429 rows × 308 columns
🔄 Duplicates: 0 (0.00%)
💾 Memory Usage: 83.98 MB


We already know that all variables after 'userscore' are the flattened output of the scraper that needs to be reformatted.  
So for now, we focus on the first 11 variables in our initial check

In [7]:
# +++ Check info on first 11 variables +++

main_vars = df.columns[:11]

df[main_vars].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13429 entries, 0 to 13428
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          13429 non-null  object
 1   genres/0       13429 non-null  object
 2   metascore      13429 non-null  object
 3   publisherName  13427 non-null  object
 4   publisherUrl   13427 non-null  object
 5   releaseDate    13397 non-null  object
 6   section        13429 non-null  object
 7   summary        13385 non-null  object
 8   type           13429 non-null  object
 9   url            13429 non-null  object
 10  userscore      13429 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [8]:
# +++ tackle the Dtypewarning from reading in the data

mixed_cols = [2, 155, 162, 163, 170, 171, 178, 179, 186, 195, 196, 197, 198]

display(df.iloc[:, mixed_cols].dtypes)

for col in df.columns[mixed_cols]:
    print(col)
    print(df[col].map(type).value_counts())

metascore                  object
platformReviews/8/name     object
platformReviews/8/url      object
platformReviews/9/name     object
platformReviews/9/url      object
platformReviews/10/name    object
platformReviews/10/url     object
platformReviews/11/name    object
platformReviews/11/url     object
platforms/8                object
platforms/9                object
platforms/10               object
platforms/11               object
dtype: object

metascore
metascore
<class 'str'>    11381
<class 'int'>     2048
Name: count, dtype: int64
platformReviews/8/name
platformReviews/8/name
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/8/url
platformReviews/8/url
<class 'float'>    13392
<class 'str'>         37
Name: count, dtype: int64
platformReviews/9/name
platformReviews/9/name
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/9/url
platformReviews/9/url
<class 'float'>    13412
<class 'str'>         17
Name: count, dtype: int64
platformReviews/10/name
platformReviews/10/name
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/10/url
platformReviews/10/url
<class 'float'>    13426
<class 'str'>          3
Name: count, dtype: int64
platformReviews/11/name
platformReviews/11/name
<class 'float'>    13428
<class 'str'>          1
Name: count, dtype: int64
platformReviews/11/url
platformReviews/11/url
<class '

It appears the we have variables that should be numerical, but are of dtype object because there are some string numbers mixed in.

In [13]:
# convert metascore to integer
df['metascore'] = pd.to_numeric(df['metascore'], errors = 'coerce').astype('Int32')

# convert userscore to float
df['userscore'] = pd.to_numeric(df['userscore'], errors = 'coerce').astype('float32')

# convert the rest of flagged variables to string
for col in df.columns[mixed_cols[1:]]:
    df[col] = df[col].astype('string')

# convert the release date to datatime format
df['releaseDate'] = pd.to_datetime(df['releaseDate'])

In [14]:
ov, num_vars, cat_vars = overview(df[main_vars])

Duplicates: 0

Auto Sales Data - Variable Overview


,dtype,total,missing_n,missing_%,uniques_n,uniques
title,object,13429,0,0.000000,13429,"[Tekken 3, Mass Effect 2, Baldur's Gate 3, The..."
genres/0,object,13429,0,0.000000,119,"[3D Fighting, Western RPG, Compilation, Linear..."
metascore,Int32,13422,7,0.052126,84,"[96, 97, 98, 99, 95, 94, 93, 92, 91, 90, 89, <..."
publisherName,object,13427,2,0.014893,2038,"[Namco, Electronic Arts, Larian Studios Games,..."
publisherUrl,object,13427,2,0.014893,2038,"[https://www.metacritic.com/company/namco/, ht..."
releaseDate,datetime64[ns],13397,32,0.238290,4761,"[1998-04-29 00:00:00, 2010-01-26 00:00:00, 202..."
section,object,13429,0,0.000000,22,"[PlayStation, Xbox 360, PC, PlayStation 3, Gam..."
summary,object,13385,44,0.327649,13314,"[An ancient evil force has reawakened, attacki..."
type,object,13429,0,0.000000,1,[game]
url,object,13429,0,0.000000,13429,"[https://www.metacritic.com/game/tekken-3, htt..."



Descriptive Metrics on numeric variables


,metascore,releaseDate,userscore
count,13422.0,13397,11896.000000
mean,70.407018,2012-08-22 01:41:08.701948160,6.929203
min,11.0,1995-04-30 00:00:00,0.300000
25%,63.0,2006-10-31 00:00:00,6.300000
50%,72.0,2012-08-08 00:00:00,7.200000
75%,79.0,2018-08-03 00:00:00,7.900000
max,99.0,2025-02-28 00:00:00,10.000000
std,12.350829,NaN,1.358217



Numeric Variables in the data set


,dtype
metascore,Int32
userscore,float32



Non-numeric variables in the data set


,dtype,n_uniques
title,object,13429
genres/0,object,119
publisherName,object,2038
publisherUrl,object,2038
section,object,22
summary,object,13314
type,object,1
url,object,13429


In [ ]:
# +++ Start the reformatting process +++

# --- Which info goes where? ---

# 1. Define metadata to repeat on every platform row.
#    Matches original column names (keys) to new ones for the new table format.
#    Will be repeated for every platform row of a game entry.
metadata = {
    "title": "title",
    "genres/0": "genre",
    "summary": "summary",
    "publisherName": "publisherName",
    "publisherUrl": "publisherUrl",
    "url": "url",
    "section": "original_platform",
    "releaseDate": "ReleaseDate",
}

# 2. How to extract the measures inside each "platformReviews/<index>/..." group.  
critic_fields = {
    "score": "metascore",
    "normalizedScore": "critic_normalized_score", # normalized score gets its own new column
    "positiveCount": "critic_positive_count",
    "negativeCount": "critic_negative_count",
    "neutralCount": "critic_mixed_count",
    "reviewCount": "critic_total_count",
}

# 3. Same as (2), but for "userReviewSummary/..." variable group
user_fields = {
    "positive": "user_positive_count",
    "negative": "user_negative_count",
    "neutral": "user_mixed_count",
    "reviewCount": "user_total_count",
}

# --- Preparation ---

# 4. Get indices from the "platformReviews/<index>/..." variable group, and sort
#    them into a integer list
indices = sorted(
    int(match.group(1))
    for column in df.columns
    if (match := re.fullmatch(r"platformReviews/(\d+)/name", str(column)))
)

# 5. Some input validation
# Are the column names unique?
if not df.columns.is_unique:
    raise ValueError("The input DataFrame must have unique column names.")
# Are there variables from "platformReview/<index>/.."group?
if not indices:
    raise ValueError("No platformReviews/<index>/name columns were found.")

# 6. Treat pandas nulls and empty strings as missing, while preserving zero.
def is_missing(value):
    return pd.isna(value) or (
        isinstance(value, str) and not value.strip()
    )

# 7. Read review measures as numbers.
#    Unavailable or nonnumeric values, such as 'tbd', become missing.
def numeric(value):
    if is_missing(value):
        return pd.NA
    return pd.to_numeric(value, errors="coerce")

# 8. Define the main output columns, including provenance for critic data.
base_columns = [
    *metadata.values(),       # all new meta-data variable names
    "platform",               # the console the current row corresponds to
    *critic_fields.values(),  # all new critic_fields variable names
    "userscore",              # user score for the console iteration of the game
    *user_fields.values(),    # all new user_fields variable names
    "critic_sourceVar",       # var group supplying critic info
    "critic_review_url",      # console critic review web page
    "userscore_sourceVar",    # source variable for user score
    "user_review_sourceVar",  # source variable for user counts
]

In [ ]:
output_rows = []

test_df = df.sample(n = 10, random_state = 42)

# 9. Process original rows sequentially
for _, game in test_df.iterrows():

    # 10. Build a dictionary containing the platform and the corresponding index
    #     as found in the "platforReviews/.." var group
    platform_slots = {}
    for index in indices:
        platform = game[f"platformReviews/{index}/name"]

        if is_missing(platform):
            continue
        # if the platform key does not exist yet, create it together with an empty list as value.
        # append the index to that list
        platform_slots.setdefault(platform, []).append(index)
    
    # 11. Create one output row for each platform found in this game.
    for platform, slots in platform_slots.items():

        # 12. Create one output row for each platform found in the data for the game        
        record = dict.fromkeys(base_columns, pd.NA) # creates dict with all base columns filled with NAs

        # fills new eta_data rows up with their respective content
        for source, destination in metadata.items():
            record[destination] = game.get(source, pd.NA)

        record['platform'] = platform

        # 13. Keep the first indexed critic observation in the main columns.
        #     Preserve later observations in matching duplicate columns,
        #     including their counts, normalized scores, URLs, and sources.
        for observation, index in enumerate(slots):
            prefix = f"platformReviews/{index}"
            suffix = (
                "" if observation == 0
                else f"_duplicate_{observation}"
            )

            for source, destination in critic_fields.items():
                record[f"{destination}{suffix}"] = numeric(
                    game.get(f"{prefix}/{source}", pd.NA)
                )

            record[f"critic_sourceVar{suffix}"] = prefix
            record[f"critic_review_url{suffix}"] = game.get(
                f"{prefix}/url", pd.NA
            )

        # 14. Check whether this is the original section platform.
        section = game.get("section", pd.NA)
        matches_section = (not is_missing(section) and platform == section) 

        if matches_section:

            # 15. Preserve the original metascore as an additional
            #     observation, even if it equals the main critic score.
            #     Number it after any repeated indexed observations.
            original_metascore = numeric(
                game.get("metascore", pd.NA)
            )

            if not is_missing(original_metascore):
                duplicate_number = len(slots)
                record[
                    f"metascore_duplicate_{duplicate_number}"
                ] = original_metascore
                record[
                    f"metascore_duplicate_{duplicate_number}_sourceVar"
                ] = "metascore"

            # 16. Assign the original userscore only to this platform.
            record["userscore"] = numeric(
                game.get("userscore", pd.NA)
            )
            record["userscore_sourceVar"] = "userscore"

            # 17. Assign user-review counts only to this platform,
            #     following our provisional section-platform assumption.
            for source, destination in user_fields.items():
                record[destination] = numeric(
                    game.get(
                        f"userReviewsSummary/{source}", pd.NA
                    )
                )

            record["user_review_sourceVar"] = "userReviewsSummary"

            # 18. Preserve the summary's user score separately,
            #     even when it equals the original userscore.
            summary_score = numeric(
                game.get("userReviewsSummary/score", pd.NA)
            )

            if not is_missing(summary_score):
                record["userscore_duplicate_1"] = summary_score
                record[
                    "userscore_duplicate_1_sourceVar"
                ] = "userReviewsSummary/score"

        # 19. Retain the platform row even if all review metrics are missing.
        output_rows.append(record)

In [151]:
output_rows

[{'title': 'Hotline Miami Collection',
  'genre': 'Compilation',
  'summary': 'Hotline Miami Collection contains both legendary games in the neon-soaked, brutally-challenging Hotline Miami series from Dennaton Games. ',
  'publisherName': 'Devolver Digital',
  'publisherUrl': 'https://www.metacritic.com/company/devolver-digital/',
  'url': 'https://www.metacritic.com/game/hotline-miami-collection',
  'original_platform': 'Nintendo Switch',
  'ReleaseDate': Timestamp('2019-08-19 00:00:00'),
  'platform': 'Nintendo Switch',
  'metascore': 82.0,
  'critic_normalized_score': 82.1923,
  'critic_positive_count': 20.0,
  'critic_negative_count': 0.0,
  'critic_mixed_count': 2.0,
  'critic_total_count': 22.0,
  'userscore': 7.699999809265137,
  'user_positive_count': 45.0,
  'user_negative_count': 9.0,
  'user_mixed_count': 8.0,
  'user_total_count': 62.0,
  'critic_sourceVar': 'platformReviews/0',
  'critic_review_url': 'https://www.metacritic.com/game/hotline-miami-collection/critic-reviews/

In [152]:
# 20. Build a new DataFrame, placing duplicate columns after main columns.
result = pd.DataFrame(output_rows)
extra_columns = [
    column for column in result.columns
    if column not in base_columns
]
result = result.reindex(columns=base_columns + extra_columns)

result.convert_dtypes()

result

,title,genre,summary,publisherName,publisherUrl,url,original_platform,ReleaseDate,platform,metascore,...,user_mixed_count,user_total_count,critic_sourceVar,critic_review_url,userscore_sourceVar,user_review_sourceVar,metascore_duplicate_1,metascore_duplicate_1_sourceVar,userscore_duplicate_1,userscore_duplicate_1_sourceVar
0,Hotline Miami Collection,Compilation,Hotline Miami Collection contains both legenda...,Devolver Digital,https://www.metacritic.com/company/devolver-di...,https://www.metacritic.com/game/hotline-miami-...,Nintendo Switch,2019-08-19,Nintendo Switch,82.0,...,8.0,62.0,platformReviews/0,https://www.metacritic.com/game/hotline-miami-...,userscore,userReviewsSummary,82.0,metascore,7.7,userReviewsSummary/score
1,Hotline Miami Collection,Compilation,Hotline Miami Collection contains both legenda...,Devolver Digital,https://www.metacritic.com/company/devolver-di...,https://www.metacritic.com/game/hotline-miami-...,Nintendo Switch,2019-08-19,PlayStation 4,<NA>,...,<NA>,<NA>,platformReviews/1,https://www.metacritic.com/game/hotline-miami-...,<NA>,<NA>,NaN,NaN,NaN,NaN
2,Hotline Miami Collection,Compilation,Hotline Miami Collection contains both legenda...,Devolver Digital,https://www.metacritic.com/company/devolver-di...,https://www.metacritic.com/game/hotline-miami-...,Nintendo Switch,2019-08-19,Xbox One,<NA>,...,<NA>,<NA>,platformReviews/2,https://www.metacritic.com/game/hotline-miami-...,<NA>,<NA>,NaN,NaN,NaN,NaN
3,Hotline Miami Collection,Compilation,Hotline Miami Collection contains both legenda...,Devolver Digital,https://www.metacritic.com/company/devolver-di...,https://www.metacritic.com/game/hotline-miami-...,Nintendo Switch,2019-08-19,PlayStation 5,<NA>,...,<NA>,<NA>,platformReviews/3,https://www.metacritic.com/game/hotline-miami-...,<NA>,<NA>,NaN,NaN,NaN,NaN
4,Hotline Miami Collection,Compilation,Hotline Miami Collection contains both legenda...,Devolver Digital,https://www.metacritic.com/company/devolver-di...,https://www.metacritic.com/game/hotline-miami-...,Nintendo Switch,2019-08-19,Xbox Series X,<NA>,...,<NA>,<NA>,platformReviews/4,https://www.metacritic.com/game/hotline-miami-...,<NA>,<NA>,NaN,NaN,NaN,NaN
5,Custom Robo,3D Fighting,Get ready to enter a world of hypertech weapon...,Nintendo,https://www.metacritic.com/company/nintendo/,https://www.metacritic.com/game/custom-robo,GameCube,2004-05-10,GameCube,65.0,...,6.0,26.0,platformReviews/0,https://www.metacritic.com/game/custom-robo/cr...,userscore,userReviewsSummary,65.0,metascore,7.8,userReviewsSummary/score
6,Summon Night: Twin Age,Action RPG,"From a very young age, the human girl Reiha di...",Atlus,https://www.metacritic.com/company/atlus/,https://www.metacritic.com/game/summon-night-t...,DS,2008-06-03,DS,73.0,...,2.0,5.0,platformReviews/0,https://www.metacritic.com/game/summon-night-t...,userscore,userReviewsSummary,73.0,metascore,6.0,userReviewsSummary/score
7,Daxter,3D Platformer,Daxter is centered around the world of the lov...,SCEA,https://www.metacritic.com/company/scea/,https://www.metacritic.com/game/daxter,PSP,2006-03-14,PSP,85.0,...,36.0,190.0,platformReviews/0,https://www.metacritic.com/game/daxter/critic-...,userscore,userReviewsSummary,85.0,metascore,8.3,userReviewsSummary/score
8,Daxter,3D Platformer,Daxter is centered around the world of the lov...,SCEA,https://www.metacritic.com/company/scea/,https://www.metacritic.com/game/daxter,PSP,2006-03-14,PlayStation 5,<NA>,...,<NA>,<NA>,platformReviews/1,https://www.metacritic.com/game/daxter/critic-...,<NA>,<NA>,NaN,NaN,NaN,NaN
9,Daxter,3D Platformer,Daxter is centered around the world of the lov...,SCEA,https://www.metacritic.com/company/scea/,https://www.metacritic.com/game/daxter,PSP,2006-03-14,PlayStation 4,<NA>,...,<NA>,<NA>,platformReviews/2,https://www.metacritic.com/game/daxter/critic-...,<NA>,<NA>,NaN,NaN,NaN,NaN
